In [1]:
from pathlib import Path
from joblib import Parallel, delayed

import torch
from tqdm.auto import tqdm
import torch

from scalesurfer.config import DEVICE, MODULE_PATH, DATA_PATH
from scalesurfer.convert import prepare_image
from scalesurfer.metrics import dense_labels_to_fs_ids, predict_volume_from_unpadded
from scalesurfer.volume.model import TransUNet3D

/home/rph/scalesurfer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from scalesurfer.experiments import build_v8_split_from_root
from scalesurfer.config import MODULE_PATH, SEED

BASE_PATH = MODULE_PATH.parent
FS_VERSION_JSON = BASE_PATH / "data" /"fs_version_by_dataset_updated.json"
TENSORS_ROOT_V8 = BASE_PATH / "data" / "tensors_gcloud"
RANDOM_SEED = int(SEED)
SPLIT_RATIOS = (0.8, 0.1, 0.1)
split_v8 = build_v8_split_from_root(
    tensors_root=TENSORS_ROOT_V8,
    seed=RANDOM_SEED,
    ratios=SPLIT_RATIOS
)

In [ ]:
from pathlib import Path
from joblib import Parallel, delayed

import torch
from tqdm.auto import tqdm
import torch

from scalesurfer.config import DEVICE, MODULE_PATH, DATA_PATH
from scalesurfer.convert import prepare_image
from scalesurfer.metrics import dense_labels_to_fs_ids, predict_volume_from_unpadded
from scalesurfer.volume.model import TransUNet3D

class ScaleSurfer:

    def __init__(
            self,
            anat_files,
            subjects,
            subject_dir,
            *,
            in_memory=False,
            n_jobs=1,
            fs_version=8,
            device=None,
            progress=True
        ):
        """
        Initilize object.

        Parameters
        ----------
        # todo: doc

        TODO: implement an efficient batch_size.
        """

        self.anat_files = anat_files
        self.subjects = subjects
        self.subject_dir = Path(subject_dir)
        self.n_jobs = n_jobs
        self.fs_version = fs_version

        self._model_volume = None
        self.device = DEVICE if device is None else device

        # todo: load based on fs_version kwarg
        self.chkpt_path_volume = MODULE_PATH.parent / "docs" / "notebooks" / "checkpoints_fsv8" / "fsv8_20260402_034229" / "transunet3d_best.pt"

        assert len(anat_files) == len(subjects), "anat_files and subjects must have the same length"
        self.prepare_directories()
        self._tqdm = tqdm if progress else lambda i: i # todo fix this is progress is False
        self.in_memory = in_memory

    @property
    def model_volume(self):
        """Lazy load volumetric model."""
        if self._model_volume is None:
            self._model_volume = TransUNet3D(
                n_classes=118,
                in_channels=1,
                base_shape=(208, 240, 192),
                patch_size=(16, 16, 16),
                channels=(12, 20, 32, 48, 64, 96),
                transformer_depth=2,
                n_heads=4,
                dropout=0.0,
                positional_encoding="sincos",
            )
            ckpt = torch.load(self.chkpt_path_volume, map_location=self.device)
            self._model_volume.load_state_dict(ckpt["model_state"])
            self._model_volume.to(self.device)
            self._model_volume.eval()
        return self._model_volume


    def model_surface(self):
        """Lazy load surface model."""
        pass

    def prepare_directories(self):
        """Create FreeSurfer-style directory structure."""
        # base dir
        self.subject_dir.mkdir(parents=True, exist_ok=True)

        for subject in self.subjects:
            # subject-level dirs
            (self.subject_dir / subject).mkdir(exist_ok=True)

            for d in ["label", "mri", "scripts", "stats", "surf", "tmp", "touch", "trash"]:
                # canonical freesurfer dirs
                (self.subject_dir / subject / d).mkdir(exist_ok=True)


    def _prepare_image(self, subject, anat_file, subject_dir):
        out_file = subject_dir / subject / "mri" / "rawavg.pt"
        if not out_file.exists():
            img_tensor = prepare_image(anat_file).to(DEVICE)
            torch.save(img_tensor, out_file)
        elif self.in_memory:
            img_tensor = torch.load(out_file)

        if self.in_memory:
            return img_tensor
        else:
            return None


    def prepare_images(self):
        img_tensors = Parallel(n_jobs=self.n_jobs)(
            delayed(self._prepare_image)(subject, anat_file, self.subject_dir)
            for subject, anat_file in tqdm(
                zip(self.subjects, self.anat_files),
                total=len(self.subjects),
                desc="Converting niftis to tensors"
            )
        )
        if self.in_memory:
            self._img_tensors = torch.stack(img_tensors) # [B, 1, D, H, W]

    def _predict_volume(self, x):
        """Predict single aparc+aseg."""
        aparc_aseg_pred = predict_volume_from_unpadded(
            model=self.model_volume,
            x_3d=x,
            patch_size=(16, 16, 16),
            device=self.device,
        )
        return aparc_aseg_pred

    def predict_volumes(self):
        """Predict aparc+aseg for all subject and save."""
        for subj in self._tqdm(self.subjects, desc="Predicting volumes"):
            if not self.in_memory:
                x = torch.load(self.subject_dir / subj / "mri"/ "rawavg.pt")
            else:
                x = self._img_tensors[self.subjects.index(subj)]
            aparc_aseg_pred = self._predict_volume(x)
            torch.save(aparc_aseg_pred, self.subject_dir / subj / "mri" / "aparc+aseg.pt")


In [76]:
import glob
# get test dataset
ds = [i[42:50] for i in split_v8["x_test"]]
nii = glob.glob("/home/rph/scalesurfer/data/openneuro_test_set_raw/files/**/*.ni*", recursive=True)
nii = {i[56:64]: i for i in nii}
ds = {i: nii[i] for i in ds}

In [77]:
images = list(ds.values())[:10]
images

['/home/rph/scalesurfer/data/openneuro_test_set_raw/files/ds002168/sub-1516/anat/sub-1516_T1w.nii.gz',
 '/home/rph/scalesurfer/data/openneuro_test_set_raw/files/ds003720/sub-005/anat/sub-005_T1w.nii',
 '/home/rph/scalesurfer/data/openneuro_test_set_raw/files/ds001241/sub-14/anat/sub-14_T1w.nii.gz',
 '/home/rph/scalesurfer/data/openneuro_test_set_raw/files/ds004589/sub-140/anat/sub-140_T1w.nii.gz',
 '/home/rph/scalesurfer/data/openneuro_test_set_raw/files/ds002878/sub-094BPAF1211010/ses-1/anat/sub-094BPAF1211010_ses-1_T1w.nii.gz',
 '/home/rph/scalesurfer/data/openneuro_test_set_raw/files/ds006188/sub-24699/ses-1/anat/sub-24699_ses-1_rec-norm_space-MNI152NLin6Asym_res-2_desc-preproc_T1w.nii.gz',
 '/home/rph/scalesurfer/data/openneuro_test_set_raw/files/ds000053/sub-017/anat/sub-017_T1w.nii.gz',
 '/home/rph/scalesurfer/data/openneuro_test_set_raw/files/ds004894/sub-05/anat/sub-05_T1w.nii.gz',
 '/home/rph/scalesurfer/data/openneuro_test_set_raw/files/ds002687/sub-1516/anat/sub-1516_T1w.nii

In [78]:
subjects = list(ds.keys())[:10]
subjects

['ds002168',
 'ds003720',
 'ds001241',
 'ds004589',
 'ds002878',
 'ds006188',
 'ds000053',
 'ds004894',
 'ds002687',
 'ds005906']

In [88]:
# Initialize object
surfer = ScaleSurfer(
    images,
    subjects,
    DATA_PATH / "subjects_dir",
    in_memory=True, # store all tensors in memory
    device="cuda"
)

In [89]:
surfer.prepare_images()

Converting niftis to tensors: 100%|██████████| 10/10 [00:00<00:00, 108.55it/s]


In [90]:
surfer.predict_volumes()

Predicting volumes: 100%|██████████| 10/10 [00:02<00:00,  3.77it/s]


In [ ]:
# todo: implement a plotting method
# surfer.plot_volume(idx) # plots image + aparc+aseg